# CS781 Phase 6 - summary length chosen per document (no training)
Reference summaries grow with the judgment (train: corr of log lengths 0.66; median 396 words for the shortest fifth of
judgments, 1,268 for the longest), but every submission so far cuts all summaries to one length. This notebook
re-cuts the **already generated** summaries of the best runs with a length that depends on the judgment's length.

1. Judgments are put in 5 length buckets (quintiles of the **training** judgments; val and test only predicted).
2. For every run, per-document BERTScore F1 (roberta-large, bert-score default for English) is computed on val at each
   length (200 / 275 / 350 / 425 words / untruncated).
3. Rule = the best length per bucket on val. Its gain is reported **honestly** with 2-fold cross-fitting on val
   (choose on one half, score on the other) next to the in-sample number.
4. A submission file is written for a run only if the cross-fitted gain over its fixed best length is positive.

**Inputs:** a dataset holding the bundles `bundle_p6_1.zip` ... `bundle_p6_10.zip` (Kaggle may unpack them on upload;
both forms are found), or the notebook outputs that contain them. Internet ON (Hugging Face data + roberta-large), secret `HF_TOKEN`, GPU T4 (one is
enough). About 15 minutes.

In [1]:
RUNS = ["D08_full_chunk", "D14_lorma_chunk", "E16_kd_chunk", "B09_full_chunk", "B15_lorma_chunk", "C17_lorma_chunk"]
LENGTHS = [200, 275, 350, 425, 0]          # 0 = untruncated (as in the training notebooks)
N_BUCKETS = 5

In [2]:
import os, glob, json, zipfile, shutil, subprocess, zlib
import numpy as np
subprocess.run("pip install -q bert-score", shell=True)
W = "/kaggle/working"; os.makedirs(f"{W}/submissions", exist_ok=True)
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"; tok_ = UserSecretsClient().get_secret("HF_TOKEN")
train = json.load(open(hf_hub_download(REPO, "Phase_6_Summ/Train/train.json", repo_type="dataset", token=tok_)))
test = json.load(open(hf_hub_download(REPO, "Phase_6_Summ/Test/test.json", repo_type="dataset", token=tok_)))
clean = lambda t: " ".join((t or "").replace(" ", " ").split())
doc_words = {str(x["id"]): len(clean(x["document"]).split()) for x in train + test}
ref = {str(x["id"]): clean(x["summary"]) for x in train}

def cut_words(t, n):
    w = t.split()
    return " ".join(w[:n]) if n else " ".join(w)

gens = {}
# bundles may arrive as zips (bundle_p6_*.zip, kept as files) or already unpacked by Kaggle (<run>/val_gen.json); take both
for b in sorted(glob.glob("/kaggle/input/**/bundle_p6_*.zip", recursive=True)):
    z = zipfile.ZipFile(b)
    names = set(z.namelist())
    for r in RUNS:
        hit = [n for n in names if n.endswith(f"{r}/val_gen.json")]
        if r in gens or not hit:
            continue
        pre = hit[0][: -len("val_gen.json")]
        if pre + "test_gen.json" in names:
            gens[r] = dict(val=json.load(z.open(pre + "val_gen.json")), test=json.load(z.open(pre + "test_gen.json")),
                           src=b)
for d in sorted(glob.glob("/kaggle/input/**/val_gen.json", recursive=True), key=len):
    r = os.path.basename(os.path.dirname(d))
    t = os.path.join(os.path.dirname(d), "test_gen.json")
    if r in RUNS and r not in gens and os.path.exists(t):
        gens[r] = dict(val=json.load(open(d)), test=json.load(open(t)), src=os.path.dirname(d))
for r in sorted(gens):
    print(f"  {r}: {gens[r]['src']}")
print("runs found:", sorted(gens), "| missing:", sorted(set(RUNS) - set(gens)))
assert gens, "attach notebook outputs that contain bundle_p6_*.zip"
val_ids = [x["id"] for x in next(iter(gens.values()))["val"]]
for r, g in gens.items():
    assert [x["id"] for x in g["val"]] == val_ids, r
val_set = set(val_ids)
test_ids = {str(x["id"]) for x in test}
for r, g in gens.items():
    assert {str(x["id"]) for x in g["test"]} == test_ids, f"{r}: test ids differ from test.json"
    assert all(x["summary"].strip() for x in g["val"] + g["test"]), f"{r}: empty summary"
    assert val_set <= set(ref), f"{r}: val ids not in train.json"
print(f"checks OK: {len(val_ids)} val documents (all in train.json), {len(test_ids)} test documents in every run")
fit_words = np.array([doc_words[str(x["id"])] for x in train if str(x["id"]) not in val_set])
edges = np.quantile(fit_words, np.linspace(0, 1, N_BUCKETS + 1)[1:-1])           # from TRAINING judgments only
bucket = lambda i: int(np.searchsorted(edges, doc_words[i], side="right"))
vb = np.array([bucket(i) for i in val_ids])
print("bucket edges (judgment words):", edges.round(0).tolist(), "| val docs per bucket:", np.bincount(vb, minlength=N_BUCKETS).tolist())

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

  B09_full_chunk: /kaggle/input/datasets/mrinaldevnath/cs781-p6-bundles/bundle_p6_4/B09_full_chunk
  B15_lorma_chunk: /kaggle/input/datasets/mrinaldevnath/cs781-p6-bundles/bundle_p6_7/B15_lorma_chunk
  C17_lorma_chunk: /kaggle/input/datasets/mrinaldevnath/cs781-p6-bundles/bundle_p6_8/C17_lorma_chunk
  D08_full_chunk: /kaggle/input/datasets/mrinaldevnath/cs781-p6-bundles/bundle_p6_4/D08_full_chunk
  D14_lorma_chunk: /kaggle/input/datasets/mrinaldevnath/cs781-p6-bundles/bundle_p6_7/D14_lorma_chunk
  E16_kd_chunk: /kaggle/input/datasets/mrinaldevnath/cs781-p6-bundles/bundle_p6_8/E16_kd_chunk
runs found: ['B09_full_chunk', 'B15_lorma_chunk', 'C17_lorma_chunk', 'D08_full_chunk', 'D14_lorma_chunk', 'E16_kd_chunk'] | missing: []
checks OK: 340 val documents (all in train.json), 100 test documents in every run
bucket edges (judgment words): [1803.0, 2644.0, 3715.0, 5596.0] | val docs per bucket: [82, 62, 64, 60, 72]


In [3]:
from bert_score import score
refs = [ref[i] for i in val_ids]
F = {}                                                                   # F[run]: (n_val, n_lengths) per-document BERTScore F1
for r, g in gens.items():
    cols = []
    for L in LENGTHS:
        cands = [cut_words(x["summary"], L) or "." for x in g["val"]]
        cols.append(score(cands, refs, lang="en", batch_size=32, verbose=False)[2].numpy())
    F[r] = np.stack(cols, 1)
    print(r, "mean BERTScore by length:", dict(zip([L or "all" for L in LENGTHS], F[r].mean(0).round(4))))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (63 outputs left out)

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C17_lorma_chunk mean BERTScore by length: {200: np.float32(0.8459), 275: np.float32(0.8485), 350: np.float32(0.8491), 425: np.float32(0.8487), 'all': np.float32(0.8486)}


In [4]:
def rule_from(Fm, b):                       # best length index per bucket (falls back to the global best)
    g = int(Fm.mean(0).argmax())
    return np.array([int(Fm[b == k].mean(0).argmax()) if (b == k).sum() >= 5 else g for k in range(N_BUCKETS)])

rng = np.random.default_rng(0)
rows, picks = [], []
for r, Fm in F.items():
    n = len(Fm); fixed = int(Fm.mean(0).argmax())
    rule = rule_from(Fm, vb)
    ins = Fm[np.arange(n), rule[vb]].mean()
    cf = []                                                               # 2-fold cross-fitting, 10 random splits
    for _ in range(10):
        p = rng.permutation(n); h = [p[: n // 2], p[n // 2:]]
        tot = 0.0
        for a, bb in ((0, 1), (1, 0)):
            ru = rule_from(Fm[h[a]], vb[h[a]]); fx = int(Fm[h[a]].mean(0).argmax())
            tot += (Fm[h[bb], ru[vb[h[bb]]]] - Fm[h[bb], fx]).sum()
        cf.append(tot / n)
    gain_cf = float(np.mean(cf))
    rows.append((r, Fm[:, fixed].mean(), LENGTHS[fixed] or "all", ins, [LENGTHS[k] or "all" for k in rule], gain_cf))
    if gain_cf > 0:
        picks.append((r, rule))
print(f"{'run':18s} {'fixed best':>10s} {'len':>4s} | {'per-bucket rule (in-sample)':>27s} | cross-fitted gain | lengths by bucket (shortest -> longest judgments)")
for r, fx, fl, ins, rl, g in rows:
    print(f"{r:18s} {fx:10.4f} {str(fl):>4s} | {ins:27.4f} | {g:+.4f}           | {rl}")

run                fixed best  len | per-bucket rule (in-sample) | cross-fitted gain | lengths by bucket (shortest -> longest judgments)
E16_kd_chunk           0.8494  425 |                      0.8496 | -0.0002           | [350, 275, 'all', 275, 350]
B09_full_chunk         0.8498  425 |                      0.8501 | -0.0001           | [350, 350, 425, 275, 275]
D08_full_chunk         0.8513  425 |                      0.8515 | -0.0001           | [425, 425, 425, 350, 350]
B15_lorma_chunk        0.8492  425 |                      0.8494 | +0.0001           | [350, 425, 425, 275, 275]
D14_lorma_chunk        0.8496  425 |                      0.8496 | -0.0003           | [425, 425, 425, 350, 275]
C17_lorma_chunk        0.8491  350 |                      0.8493 | -0.0004           | [350, 350, 425, 350, 275]


In [5]:
written = []
for r, rule in picks:
    pred = [{"id": x["id"], "summary": cut_words(x["summary"], LENGTHS[rule[bucket(x["id"])]]) or "."} for x in gens[r]["test"]]
    assert len(pred) == len(test) and len({p["id"] for p in pred}) == len(test)
    name = f"LEN_{r}"
    with zipfile.ZipFile(f"{W}/submissions/{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(pred, ensure_ascii=False))
    written.append(name)
print("submission files (only runs whose cross-fitted gain is positive):", written or "none - keep the fixed lengths")
json.dump(dict(edges=edges.tolist(), rows=[dict(run=r, fixed=float(fx), fixed_len=str(fl), rule_in_sample=float(ins),
               rule_lengths=[str(x) for x in rl], gain_crossfit=g) for r, fx, fl, ins, rl, g in rows]),
          open(f"{W}/length_rule.json", "w"), indent=1)

submission files (only runs whose cross-fitted gain is positive): ['LEN_B15_lorma_chunk']
